# Comparación de Metodologías de Trayectorias

El objetivo es comparar la **metodología nueva (v1/v2)** con la **metodología anterior** (del `notebook 01`) para entender:
- Qué cambió realmente.
- Ganancias y pérdidas.
- Sesgos detectados.


In [8]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import ast
from IPython.display import display

# Cargar base limpia para reproducir episodios
df_base = pd.read_excel("../data/final_data/df_base_limpia.xlsx")

# ==============================================================
# 1. RECONSTRUIR OUTPUT DE LA METODOLOGÍA ANTERIOR
# ==============================================================
# Ordenamos por paciente y fecha
df_estancias_episodios = df_base.sort_values(['paciente_id', 'fecha_ingreso']).copy()

# Agrupamos como se hacía en 01_redes_basico.ipynb
df_rutas = df_estancias_episodios.groupby('paciente_id').agg(
    trayectoria_hospitalaria=('hospital_origen', lambda x: str(list(x))), 
    hospital_inicio=('hospital_origen', 'first'),
    hospital_final=('hospital_origen', 'last'),
    n_hospitales_unicos=('hospital_origen', lambda x: len(set(x))),
    n_episodios=('hospital_origen', 'count')
).reset_index()

df_old = df_rutas.copy()

In [9]:
# 2. ALINEAR ESTRUCTURAS
# ==============================================================
df_v1 = pd.read_excel("../data/final_data/df_pacientes_trayectorias_v1.xlsx")
df_v2 = pd.read_excel("../data/final_data/df_pacientes_trayectorias_v2.xlsx")

def rename_cols(df, suffix):
    cols = {
        'trayectoria_hospitalaria': f'trayectoria{suffix}',
        'hospital_inicio': f'inicio{suffix}',
        'hospital_final': f'final{suffix}',
        'n_hospitales_unicos': f'n_unicos{suffix}',
        'n_episodios': f'n_episodios{suffix}'
    }
    return df.rename(columns={k: v for k, v in cols.items() if k in df.columns})

df_old = rename_cols(df_old, '_old')
df_v1 = rename_cols(df_v1, '_v1')
df_v2 = rename_cols(df_v2, '_v2')

# Unir en una mega tabla
df_comp = df_old.merge(df_v1, on='paciente_id', how='outer')\
                .merge(df_v2, on='paciente_id', how='outer')


In [10]:
# ==============================================================
# 3. COMPARACIÓN DE COBERTURA
# ==============================================================
print("=== COBERTURA ===")
print(f"Pacientes en Old: {df_old['paciente_id'].nunique()}")
print(f"Pacientes en V1:  {df_v1['paciente_id'].nunique()}")
print(f"Pacientes en V2:  {df_v2['paciente_id'].nunique()}")

comunes = df_comp.dropna(subset=['trayectoria_old', 'trayectoria_v1', 'trayectoria_v2'])
print(f"\nPacientes en común (presentes en las 3): {len(comunes)}")

perdidos = df_comp[df_comp['trayectoria_v1'].isna() & df_comp['trayectoria_old'].notna()]
print(f"Pacientes perdidos (estaban en Old pero no en V1/V2): {len(perdidos)} (por los filtros de limpieza)")


=== COBERTURA ===
Pacientes en Old: 22729
Pacientes en V1:  19980
Pacientes en V2:  19980

Pacientes en común (presentes en las 3): 19980
Pacientes perdidos (estaban en Old pero no en V1/V2): 2749 (por los filtros de limpieza)


In [11]:
# ==============================================================
# 4. COMPARACIÓN DE TRAYECTORIAS
# ==============================================================
df_comunes = comunes.copy()

# Igualdad exacta (strings vs strings)
df_comunes['exact_old_v1'] = df_comunes['trayectoria_old'] == df_comunes['trayectoria_v1']
df_comunes['exact_old_v2'] = df_comunes['trayectoria_old'] == df_comunes['trayectoria_v2']
df_comunes['exact_v1_v2']  = df_comunes['trayectoria_v1'] == df_comunes['trayectoria_v2']

print("=== EXACTITUD DE TRAYECTORIAS ===")
print(f"Old igual a V1: {df_comunes['exact_old_v1'].mean():.2%}")
print(f"Old igual a V2: {df_comunes['exact_old_v2'].mean():.2%}")
print(f"V1 igual a V2:  {df_comunes['exact_v1_v2'].mean():.2%}")


=== EXACTITUD DE TRAYECTORIAS ===
Old igual a V1: 99.48%
Old igual a V2: 99.69%
V1 igual a V2:  99.59%


In [12]:
# ==============================================================
# 5. COMPARACIÓN DE RESULTADOS CLAVE
# ==============================================================
print("=== MÉTRICAS CLAVE (% que cambia respecto a Old) ===")
cambio_final_v1 = (df_comunes['final_old'] != df_comunes['final_v1']).mean()
cambio_final_v2 = (df_comunes['final_old'] != df_comunes['final_v2']).mean()
print(f"Hospital Final distinto a Old -> V1: {cambio_final_v1:.2%} | V2: {cambio_final_v2:.2%}")

cambio_unicos_v1 = (df_comunes['n_unicos_old'] != df_comunes['n_unicos_v1']).mean()
cambio_unicos_v2 = (df_comunes['n_unicos_old'] != df_comunes['n_unicos_v2']).mean()
print(f"Hospitales Únicos distinto a Old -> V1: {cambio_unicos_v1:.2%} | V2: {cambio_unicos_v2:.2%}")


=== MÉTRICAS CLAVE (% que cambia respecto a Old) ===
Hospital Final distinto a Old -> V1: 0.30% | V2: 0.00%
Hospitales Únicos distinto a Old -> V1: 0.14% | V2: 0.00%


In [13]:
# ==============================================================
# 6. ANÁLISIS DE CASOS DIFERENTES (CLAVE)
# ==============================================================
casos_A = df_comunes[~df_comunes['exact_old_v1']]
casos_B = df_comunes[~df_comunes['exact_old_v2']]
casos_C = df_comunes[~df_comunes['exact_v1_v2']]

print("=== CONTEO DE CONFLICTOS ===")
print(f"🔴 Caso A (Old != V1): {len(casos_A)} pacientes")
print(f"🔵 Caso B (Old != V2): {len(casos_B)} pacientes")
print(f"🟣 Caso C (V1 != V2):  {len(casos_C)} pacientes")


=== CONTEO DE CONFLICTOS ===
🔴 Caso A (Old != V1): 103 pacientes
🔵 Caso B (Old != V2): 61 pacientes
🟣 Caso C (V1 != V2):  81 pacientes


In [14]:
# ==============================================================
# MUESTRA DETALLADA DE CONFLICTOS
# ==============================================================
def mostrar_ejemplo(paciente_id, label):
    row = df_comunes[df_comunes['paciente_id'] == paciente_id].iloc[0]
    print("\n" + "="*80)
    print(f"EJEMPLO: {label} | PACIENTE ID: {paciente_id}")
    print("-"*80)
    print(f"OLD: {row['trayectoria_old']}")
    print(f"V1:  {row['trayectoria_v1']}")
    print(f"V2:  {row['trayectoria_v2']}")
    print("\n👉 Episodios Originales:")
    ep_cols = ['fecha_ingreso', 'fecha_egreso', 'hospital_origen', 'motivo_egreso']
    display(df_base[df_base['paciente_id'] == paciente_id][ep_cols])

# Mostrar todos los IDs que difieren en V1 vs V2
print("\n🟣 LISTADO COMPLETO DE PACIENTES DONDE V1 != V2:")
display(casos_C['paciente_id'].tolist()[:50]) # limitamos a 50 visuales por si son muchos

# Mostrar samples detallados
if not casos_A.empty: mostrar_ejemplo(casos_A['paciente_id'].sample(1, random_state=42).iloc[0], "🔴 Caso A (Old != V1)")
if not casos_B.empty: mostrar_ejemplo(casos_B['paciente_id'].sample(1, random_state=43).iloc[0], "🔵 Caso B (Old != V2)")

# Mostrar unos cuantos de V1 != V2
print("\n\n🟣 SAMPLES DE V1 != V2:")
for p_id in casos_C['paciente_id'].sample(min(5, len(casos_C)), random_state=44).tolist():
    mostrar_ejemplo(p_id, "Caso C (V1 != V2)")



🟣 LISTADO COMPLETO DE PACIENTES DONDE V1 != V2:


['AA31',
 'AB52',
 'AO28',
 'AQ19',
 'AR06',
 'BA75',
 'BH66',
 'BJ81',
 'BW02',
 'BY84',
 'CE21',
 'CF03',
 'CI83',
 'CY84',
 'DD32',
 'DG58',
 'DI69',
 'DI98',
 'DN11',
 'DP85',
 'FG00',
 'FX21',
 'FZ99',
 'GM66',
 'HB69',
 'HC81',
 'HX17',
 'IG07',
 'IM89',
 'IO09',
 'IX46',
 'JC07',
 'JS30',
 'JS87',
 'KK11',
 'KL75',
 'KM44',
 'LE46',
 'LI01',
 'LN35',
 'MF69',
 'MM79',
 'MO94',
 'NC75',
 'NV93',
 'OD33',
 'OM49',
 'OP83',
 'PA09',
 'PB20']


EJEMPLO: 🔴 Caso A (Old != V1) | PACIENTE ID: HB69
--------------------------------------------------------------------------------
OLD: ['Módulo Hospitalario 9 - AB', 'UPA 5 - AB', 'Lucio Meléndez']
V1:  ['UPA 5 - AB', 'Lucio Meléndez', 'UPA 5 - AB']
V2:  ['Módulo Hospitalario 9 - AB', 'UPA 5 - AB', 'Lucio Meléndez']

👉 Episodios Originales:


,fecha_ingreso,fecha_egreso,hospital_origen,motivo_egreso
6827,2021-07-16 12:11:21,2021-07-20 12:11:21,Módulo Hospitalario 9 - AB,traslado-hospital-de-la-red
6828,2021-07-16 12:11:25,2021-07-16 12:11:25,UPA 5 - AB,traslado-hospital-de-la-red
6829,2021-07-20 12:11:16,2021-07-22 12:11:16,Lucio Meléndez,alta-domiciliaria



EJEMPLO: 🔵 Caso B (Old != V2) | PACIENTE ID: WN04
--------------------------------------------------------------------------------
OLD: ['UPA 5 - AB', 'Módulo Hospitalario 9 - AB', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB']
V1:  ['UPA 5 - AB', 'Módulo Hospitalario 9 - AB']
V2:  ['UPA 5 - AB', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB']

👉 Episodios Originales:


...

,fecha_ingreso,fecha_egreso,hospital_origen,motivo_egreso
24376,2021-04-21 12:11:08,2021-04-29 12:11:08,Mi Pueblo,traslado-hospital-de-la-red
24377,2021-04-29 12:11:05,2021-04-29 12:11:05,El Cruce,traslado-hospital-de-la-red
24378,2021-04-29 12:11:08,2021-04-30 12:11:08,Mi Pueblo,traslado-hospital-de-la-red
24379,2021-04-30 12:11:05,2021-05-18 12:11:05,El Cruce,muerte



EJEMPLO: Caso C (V1 != V2) | PACIENTE ID: OP83
--------------------------------------------------------------------------------
OLD: ['Lucio Meléndez', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB']
V1:  ['Módulo Hospitalario 9 - AB', 'UPA 5 - AB']
V2:  ['Lucio Meléndez', 'Módulo Hospitalario 9 - AB', 'UPA 5 - AB']

👉 Episodios Originales:


,fecha_ingreso,fecha_egreso,hospital_origen,motivo_egreso
13953,2022-06-07 12:11:16,2022-06-09 12:11:16,Lucio Meléndez,alta-domiciliaria
13954,2022-06-07 12:11:21,2022-06-07 12:11:21,Módulo Hospitalario 9 - AB,traslado-hospital-de-la-red
13955,2022-06-07 12:11:25,2022-06-07 12:11:25,UPA 5 - AB,traslado-hospital-de-la-red



EJEMPLO: Caso C (V1 != V2) | PACIENTE ID: ZZ37
--------------------------------------------------------------------------------
OLD: ['UPA 11 - FV', 'Módulo Hospitalario 11 - FV', 'Módulo Hospitalario 11 - FV', 'UPA 11 - FV']
V1:  ['UPA 11 - FV', 'Módulo Hospitalario 11 - FV']
V2:  ['UPA 11 - FV', 'Módulo Hospitalario 11 - FV', 'UPA 11 - FV']

👉 Episodios Originales:


,fecha_ingreso,fecha_egreso,hospital_origen,motivo_egreso
24918,2021-04-15 12:11:26,2021-04-17 12:11:26,UPA 11 - FV,traslado-hospital-de-la-red
24919,2021-04-17 12:11:10,2021-04-17 12:11:10,Módulo Hospitalario 11 - FV,traslado-hospital-de-la-red
24920,2021-04-17 12:11:10,2021-05-25 12:11:10,Módulo Hospitalario 11 - FV,muerte
24921,2021-04-17 12:11:26,2021-04-17 12:11:26,UPA 11 - FV,traslado-hospital-de-la-red


## 7. INTERPRETACIÓN Y CONCLUSIÓN

### 7.1 Sobre la metodología vieja
* **¿Qué estaba asumiendo sin hacerlo explícito?** Asumía que toda aparición de un paciente en un hospital formaba parte de una cadena conectada de eventos. Si un paciente iba a la guardia, se iba de alta y volvía 3 años después a otro hospital, la base vieja lo consideraba una "trayectoria" (A -> B). Además, no colapsaba registros repetidos consecutivos causados por burocracia (A -> A).
* **¿Dónde puede fallar?** Altera severamente los gráficos de red al inyectar "flujos" (aristas) que no son traslados reales, inflando las conexiones entre hospitales.

### 7.2 Sobre v1 (Traslados)
* **¿Qué mejora?** Aisla la **verdadera red de derivaciones logísticas**. Obliga a que la transición esté respaldada por una ventana temporal minúscula (o un motivo de traslado explícito), eliminando saltos temporales absurdos.
* **¿Qué pierde?** Puede perder conexiones donde administrativamente un hospital dio el "alta" en vez del "traslado" y el paciente tardó varios días en ingresar al destino, clasificándolo erróneamente como dos episodios inconexos.

### 7.3 Sobre v2 (Episodios)
* **¿Qué mantiene de la vieja?** Mantiene la secuencia cronológica pura sin exigir validación de traslados logísticos.
* **¿Qué cambia?** Ahora es explícita y levanta *flags* (alertas de gaps y overlaps) además de colapsar eventos A -> A repetidos, depurando la visualización cronológica.

### 7.4 Conclusión Final
* **¿Cuál representa mejor la realidad de la red hospitalaria?** Para análisis de flujos, cuellos de botella y mapas logísticos, la **V1 es inmensamente superior**. Evita graficar conexiones donde no hubo ambulancia ni transferencia real de pacientes.
* **¿Cuál es más robusta a errores administrativos?** **V1**. Mitiga las altas mal codificadas al no asumir conexiones espontáneas de largo plazo.
* **Veredicto:** Combinar. Usar **V1** como "Ground Truth" para cualquier cálculo de "Derivación en Red", y usar **V2 filtrando aquellos con flags de gap grande** para un análisis puro de reingresos crónicos a largo plazo.
